> **Original course-project notebook (2026), kept for reference; superseded by `src/dfer/`.** Outputs are cleared: they contained DFEW frames (licence-restricted) and results that were selected on the fold's test set and affected by a frame-ordering bug. See the README for the corrected pipeline and results.

#Self-Supervised Learning with V-JEPA for DFER

In this notebook, we implement V-JEPA, a self-supervised model for video classification. The model is pretrained on a large generic video dataset (not Kinetics) and fine-tuned on the DFEW dataset for dynamic facial expression recognition (DFER).

We perform the following steps:

* Load and preprocess the DFEW dataset.
* Fine-tune the V-JEPA model, which is a latent prediction model.
* Implement two stages of training:
    * Stage 1: Fine-tune the classifier only (freeze encoder).
    * Stage 2: Fine-tune the encoder (partial fine-tuning).
* Evaluate performance based on accuracy and UAR (Unweighted Average Recall).

This setup helps us compare the effectiveness of self-supervised learning using a model like V-JEPA in a real-world emotion recognition task.

##Imports and Setup

In this section, we import the necessary libraries for data processing, training, and evaluation:

* PyTorch: For model building and training.
* Transformers: To load the V-JEPA model.
* Matplotlib and tqdm: For visualizations and progress bars.
* YAML: For loading configuration files.

We then define the root directories for the project, dataset paths, and output directories for saving results, logs, and checkpoints.

In [ ]:
import os, sys, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from tqdm import tqdm
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

from transformers import VJEPA2ForVideoClassification
import yaml

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

CONFIG_PATH = PROJECT_ROOT / "configs" / "paths.yaml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

DFEW_ROOT = config["dfew_root"]
SPLITS_ROOT = config["splits_root"]

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
LOG_DIR    = OUTPUT_DIR / "logs"
PLOT_DIR   = OUTPUT_DIR / "plots"

for d in [OUTPUT_DIR, CKPT_DIR, LOG_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DFEW:", DFEW_ROOT)

##Reproducibility and Device Setup

To ensure reproducibility across runs, we set a fixed random seed. This ensures consistent results each time the experiment is run.

We also check if a GPU (CUDA) is available for training. If available, we use CUDA for faster training; otherwise, we fall back to the CPU.

We define the following training parameters:

* NUM_FRAMES: Number of frames per video.
* IMG_SIZE: The input image size for each video frame.
* BATCH_SIZE: The number of videos processed per batch.
* EPOCHS_STAGE1 and EPOCHS_STAGE2: Number of epochs for each stage of training (freeze head vs fine-tune entire model).

In [ ]:
SEED = 42
FOLD = 1

NUM_FRAMES = 16
IMG_SIZE = 224
NUM_CLASSES = 7

BATCH_SIZE = 4
NUM_WORKERS = 4
PIN_MEMORY = True

EPOCHS_STAGE1 = 6
EPOCHS_STAGE2 = 14

PATIENCE = 5
MIN_DELTA = 1e-4

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

RUN_NAME = f"jepa_fold{FOLD}_FINAL"

TRAIN_CSV = os.path.join(SPLITS_ROOT, "train(single-labeled)", f"set_{FOLD}.csv")
VAL_CSV   = os.path.join(SPLITS_ROOT, "test(single-labeled)",  f"set_{FOLD}.csv")

##Data Loading and Transformation

In this section, we define the data transformations for both the train and validation datasets:

* Training transformations: Random resizing and horizontal flipping to augment the dataset.
* Validation transformations: Resizing and normalization to prepare the data for evaluation.

We then load the train and validation datasets using the DFEWVideoDataset class, which reads from the provided CSV files. The DataLoader is used to handle batching and shuffling.

In [ ]:
from dataset import DFEWVideoDataset

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

train_ds = DFEWVideoDataset(DFEW_ROOT, TRAIN_CSV, train_transform, NUM_FRAMES)
val_ds   = DFEWVideoDataset(DFEW_ROOT, VAL_CSV,   val_transform,   NUM_FRAMES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))

##Class Weights for Imbalance

In this section, we calculate class weights to handle the class imbalance in the DFEW dataset:

* Rarer classes are given higher weights to ensure they are properly learned during training.
* We compute the inverse of the frequency of each class and normalize it.

The calculated class weights are stored in a tensor and moved to the appropriate device (GPU/CPU).

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
train_counts = train_df["label"].value_counts().sort_index()

freq = train_counts.values.astype(np.float32)
weights = 1.0 / (freq + 1e-6)
weights = weights / weights.sum() * len(weights)

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

##Model Setup: V-JEPA

In this section, we load the V-JEPA model pretrained on a large generic video dataset:

* facebook/vjepa2-vitl-fpc64-256 is the pretrained model that we will fine-tune for emotion classification on the DFEW dataset.
* The model is configured to handle 7 classes for emotion classification.

We load the model using the Transformers library and move it to the appropriate device (GPU/CPU).

In [ ]:
MODEL_ID = "facebook/vjepa2-vitl-fpc64-256"

model = VJEPA2ForVideoClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(device)

print("Loaded:", MODEL_ID)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None

##Training and Validation Loops

In [ ]:
def train_model(num_epochs, stage_name, optimizer, scheduler):

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "val_uar": [],
        "lr": [],
        "epoch_time_min": []
    }

    best_val_acc = -1.0
    best_val_uar = -1.0
    no_improve_uar = 0

    best_acc_path = CKPT_DIR / f"{RUN_NAME}_{stage_name}_best_acc.pth"
    best_uar_path = CKPT_DIR / f"{RUN_NAME}_{stage_name}_best_uar.pth"
    history_csv   = LOG_DIR / f"{RUN_NAME}_{stage_name}_history.csv"

    for epoch in range(num_epochs):
        epoch_start = time.time()

        # ---------------- TRAIN ----------------
        model.train()
        train_losses, train_preds, train_targets = [], [], []

        pbar = tqdm(train_loader, desc=f"{stage_name} Epoch {epoch+1}/{num_epochs}", leave=True)

        for videos, labels in pbar:
            if videos.dim() == 5 and videos.shape[1] == 3:
                videos = videos.permute(0, 2, 1, 3, 4)

            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if device == "cuda":
                with torch.amp.autocast("cuda"):
                    outputs = model(pixel_values_videos=videos)
                    logits = outputs.logits
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(pixel_values_videos=videos)
                logits = outputs.logits
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()

            train_losses.append(loss.item())
            preds = logits.argmax(1).detach().cpu().numpy()
            train_preds.extend(preds)
            train_targets.extend(labels.detach().cpu().numpy())

            pbar.set_postfix({
                "loss": float(np.mean(train_losses)),
                "acc": float(accuracy_score(train_targets, train_preds)),
                "lr": float(optimizer.param_groups[0]["lr"])
            })

        train_loss = float(np.mean(train_losses))
        train_acc  = float(accuracy_score(train_targets, train_preds))

        # ---------------- VAL ----------------
        model.eval()
        val_losses, val_preds, val_targets = [], [], []

        with torch.no_grad():
            for videos, labels in val_loader:
                if videos.dim() == 5 and videos.shape[1] == 3:
                    videos = videos.permute(0, 2, 1, 3, 4)

                videos = videos.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                outputs = model(pixel_values_videos=videos)
                logits = outputs.logits
                loss = criterion(logits, labels)

                val_losses.append(loss.item())
                val_preds.extend(logits.argmax(1).cpu().numpy())
                val_targets.extend(labels.cpu().numpy())

        val_loss = float(np.mean(val_losses))
        val_acc  = float(accuracy_score(val_targets, val_preds))
        val_uar  = float(recall_score(val_targets, val_preds, average="macro"))

        scheduler.step()

        lr_now = float(optimizer.param_groups[0]["lr"])
        epoch_time = (time.time() - epoch_start) / 60.0

        print(f"\n[{stage_name}] Epoch {epoch+1}/{num_epochs}")
        print(f"Train Loss {train_loss:.4f} | Acc {train_acc:.4f}")
        print(f"Val   Loss {val_loss:.4f} | Acc {val_acc:.4f} | UAR {val_uar:.4f}")
        print(f"LR {lr_now:.6f} | Time {epoch_time:.2f} min")

        history["epoch"].append(epoch+1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_uar"].append(val_uar)
        history["lr"].append(lr_now)
        history["epoch_time_min"].append(epoch_time)

        pd.DataFrame(history).to_csv(history_csv, index=False)

        # BEST-ACC
        if val_acc > best_val_acc + MIN_DELTA:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_acc_path)
            print("New BEST-ACC saved.")

        # BEST-UAR (PRIMARY) + early stopping on UAR
        if val_uar > best_val_uar + MIN_DELTA:
            best_val_uar = val_uar
            torch.save(model.state_dict(), best_uar_path)
            no_improve_uar = 0
            print("New BEST-UAR saved.")
        else:
            no_improve_uar += 1
            print(f"No UAR improvement {no_improve_uar}/{PATIENCE}")

        if no_improve_uar >= PATIENCE:
            print("Early stopping (UAR).")
            break

    return history, best_acc_path, best_uar_path

##Stage 1: Fine-Tuning the Classifier

In this section, we fine-tune the classifier part of the V-JEPA model (keeping the encoder frozen). We will only train the last classification layer, while the rest of the encoder remains unchanged.

The hyperparameters for this stage:

* Learning Rate: Set to 1e-3 for the classifier.
* Weight Decay: Regularization to prevent overfitting.
* Cosine Annealing Scheduler: Adjusts the learning rate during training.

The objective is to fine-tune the model’s head to adapt to the DFEW dataset while keeping the encoder frozen.

In [ ]:
for p in model.vjepa2.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True

optimizer_stage1 = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

scheduler_stage1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_stage1,
    T_max=EPOCHS_STAGE1,
    eta_min=1e-6
)

history_stage1, best_acc_stage1, best_uar_stage1 = train_model(
    EPOCHS_STAGE1, "stage1_head", optimizer_stage1, scheduler_stage1
)

##Stage 2: Fine-Tuning the Encoder (Partial Fine-Tuning)

In this stage, we unfreeze the last few layers of the encoder and fine-tune them along with the classifier. This allows the model to adapt more effectively to the specific task of emotion classification.

We use the following settings:

* Learning Rate: A smaller learning rate of 3e-5 for the encoder layers and 5e-4 for the classifier.
* Cosine Annealing Scheduler: Adjusts the learning rate during training.
* Weight Decay: Regularization to prevent overfitting.

The goal is to see how well the model can improve by adapting the encoder after fine-tuning the classifier in the first stage.

In [ ]:
for p in model.vjepa2.parameters():
    p.requires_grad = False

for block in model.vjepa2.encoder.layer[-2:]:
    for p in block.parameters():
        p.requires_grad = True

for p in model.classifier.parameters():
    p.requires_grad = True

optimizer_stage2 = torch.optim.AdamW([
    {"params": model.vjepa2.encoder.layer[-2:].parameters(), "lr": 3e-5},  # stronger than before
    {"params": model.classifier.parameters(), "lr": 5e-4},
], weight_decay=0.05)

scheduler_stage2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_stage2,
    T_max=EPOCHS_STAGE2,
    eta_min=1e-6
)

history_stage2, best_acc_stage2, best_uar_stage2 = train_model(
    EPOCHS_STAGE2, "stage2_partial", optimizer_stage2, scheduler_stage2
)

##PLots (Acc, Loss , UAR) (Stage1 , Stage2)

In [ ]:
def save_plots(hist, name):
    df = pd.DataFrame(hist)

    plt.figure()
    plt.plot(df["epoch"], df["train_loss"], label="train")
    plt.plot(df["epoch"], df["val_loss"], label="val")
    plt.legend(); plt.title(name + " Loss"); plt.grid()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{RUN_NAME}_{name}_loss.png", dpi=200)
    plt.show()

    plt.figure()
    plt.plot(df["epoch"], df["train_acc"], label="train")
    plt.plot(df["epoch"], df["val_acc"], label="val")
    plt.legend(); plt.title(name + " Accuracy"); plt.grid()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{RUN_NAME}_{name}_accuracy.png", dpi=200)
    plt.show()

    plt.figure()
    plt.plot(df["epoch"], df["val_uar"], label="val_uar")
    plt.legend(); plt.title(name + " UAR"); plt.grid()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{RUN_NAME}_{name}_uar.png", dpi=200)
    plt.show()

save_plots(history_stage1, "stage1_head")
save_plots(history_stage2, "stage2_partial")

##Confusion Matrices (Acc , UAR) (Stage2)

In [ ]:
def eval_cm(weights_path, title, save_path):
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.to(device)
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for videos, labels in tqdm(val_loader, desc=title, leave=False):
            if videos.dim() == 5 and videos.shape[1] == 3:
                videos = videos.permute(0, 2, 1, 3, 4)

            videos = videos.to(device)
            logits = model(pixel_values_videos=videos).logits

            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_targets.extend(labels.numpy())

    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(6,6))
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()

    print(title,
          "Acc:", accuracy_score(all_targets, all_preds),
          "UAR:", recall_score(all_targets, all_preds, average="macro"))

eval_cm(best_acc_stage2,
        f"{RUN_NAME} BEST ACC (stage2)",
        PLOT_DIR / f"{RUN_NAME}_cm_best_acc.png")

eval_cm(best_uar_stage2,
        f"{RUN_NAME} BEST UAR (stage2)",
        PLOT_DIR / f"{RUN_NAME}_cm_best_uar.png")